# Israel Kupot Cholim Doctor Data - Analysis Notebook

This notebook provides exploratory analysis of scraped doctor data from Israeli health funds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data/processed')
RAW_DATA_DIR = Path('../data/raw')

KUPA_NAMES = {
    'clalit': 'Clalit',
    'maccabi': 'Maccabi',
    'meuhedet': 'Meuhedet',
    'leumit': 'Leumit'
}

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## Load Data

In [ ]:
def load_doctor_data(year=None):
    """Load doctor data for a specific year or the most recent."""
    if year:
        filepath = DATA_DIR / f'all_doctors_{year}.csv'
    else:
        files = list(DATA_DIR.glob('all_doctors_*.csv'))
        if not files:
            print("No data files found")
            return None
        filepath = sorted(files)[-1]
    
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    print(f"Loaded {len(df)} records from {filepath.name}")
    return df

df = load_doctor_data()

## Basic Statistics

In [ ]:
def basic_stats(df):
    """Show basic statistics about the data."""
    print("="*50)
    print("BASIC STATISTICS")
    print("="*50)
    print(f"Total records: {len(df)}")
    print(f"Unique doctors: {df['name'].nunique()}")
    print(f"\nBy Gender:")
    print(df['gender'].value_counts())
    print(f"\nBy Specialty (top 10):")
    print(df['specialty'].value_counts().head(10))

basic_stats(df)

## Kupat Cholim Distribution

In [ ]:
def kupa_distribution(df):
    """Show distribution of doctors across Kupot Cholim."""
    kupa_cols = ['clalit', 'maccabi', 'meuhedet', 'leumit']
    
    counts = {KUPA_NAMES[k]: df[k].sum() for k in kupa_cols}
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].bar(counts.keys(), counts.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    axes[0].set_title('Doctors by Kupat Cholim')
    axes[0].set_ylabel('Number of Doctors')
    for i, (k, v) in enumerate(counts.items()):
        axes[0].text(i, v + 50, str(v), ha='center')
    
    axes[1].pie(counts.values(), labels=counts.keys(), autopct='%1.1f%%', 
                colors=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    axes[1].set_title('Distribution of Doctors')
    
    plt.tight_layout()
    plt.show()
    
    return counts

kupa_dist = kupa_distribution(df)

## Dual Practice Analysis

Doctors working in multiple Kupot Cholim (dual practice).

In [ ]:
def dual_practice_analysis(df):
    """Analyze doctors working in multiple Kupot Cholim."""
    kupa_cols = ['clalit', 'maccabi', 'meuhedet', 'leumit']
    
    df['kupa_count'] = df[kupa_cols].sum(axis=1)
    
    dual_practice = df[df['kupa_count'] > 1]
    print(f"Doctors in multiple Kupot: {len(dual_practice)} ({len(dual_practice)/len(df)*100:.1f}%)")
    
    print("\nBreakdown by number of Kupot:")
    print(df['kupa_count'].value_counts().sort_index())
    
    return dual_practice

dual_docs = dual_practice_analysis(df)

## Atzmai (Self-Employed) Identification

This section helps identify doctors who work as self-employed (atzmai) 
which is the main challenge for categorizing income as public vs private.

In [ ]:
def identify_atzmai_indicators(df):
    """Identify potential atzmai (self-employed) doctors."""
    
    atzmai_keywords = ['עצמאי', 'אישי', 'פרטי', 'atzmai', 'self']
    
    if 'clinic_address' in df.columns:
        address_pattern = df['clinic_address'].str.contains('|'.join(atzmai_keywords), 
                                                        case=False, na=False)
        df['potential_atzmai'] = address_pattern
    
    return df

df = identify_atzmai_indicators(df)

## Export Data

In [ ]:
def export_for_analysis(df, output_file='doctors_for_salary_analysis.csv'):
    """Export data formatted for salary analysis."""
    
    export_df = df[['name', 'specialty', 'gender', 'year', 
                    'clalit', 'maccabi', 'meuhedet', 'leumit',
                    'license_number', 'source_name']].copy()
    
    export_path = DATA_DIR / output_file
    export_df.to_csv(export_path, index=False, encoding='utf-8-sig')
    print(f"Exported to {export_path}")
    
    return export_df

export_df = export_for_analysis(df)